# 09 휘도비 등급 경계: 문헌 기준 vs 사고 데이터 기준

- ① 문헌: 휘도비 10배 미만 하 / 10~15배 중 / 15배 초과 상 (Schreuder 1991)
- ② 사고 데이터: 2023~2024 사고로 후보 경계 중 가장 잘 가르는 것
- 비교: 2025 사고(찾을 때 안 씀)에서 llf(클수록 좋음)와 배수
- 사고 vs 같은 자리·같은 요일·시각 다른 주(07과 같은 case-crossover), 낮만
- 순응도 A·등급 기준은 초안 · 업데이트 중 (재혁 검토 전)

In [ ]:
import pandas as pd, numpy as np, warnings
from pathlib import Path
from statsmodels.discrete.conditional_models import ConditionalLogit
warnings.filterwarnings('ignore')

DATA = Path('data')
t = pd.read_csv(DATA / '3_결과' / 'train_table.csv', low_memory=False)
ZONES = ['진입전', '경계부', '출구부', '진출후']
cc = t[t.design.isin(['사고', '같은지점']) & t.tunnel_zone.isin(ZONES)]
cc = cc[['case_id', 'y', 't', 'design', 'tunnel_zone', 'sun_elev', 'rn', 'icsr']]
g = pd.read_csv(DATA / '2_중간' / 'gk2a_solar.csv')[['case_id', 'y', 't', 'gk2a_icsr']]
cc = cc.merge(g, on=['case_id', 'y', 't'], how='left')
cc['t'] = pd.to_datetime(cc.t)


In [ ]:
cc=cc[cc.case_id.isin(cc[cc.y==0].case_id)]
cc['rain']=(cc.rn.fillna(0)>0).astype(int)
ENT=['진입전','경계부']; EXT=['출구부','진출후']
def ratio(icsr,zone):
    L20=0.15*(icsr*277.8*126)/np.pi
    step=np.clip(np.ceil(L20/4000*4)/4,0.25,1)
    return np.where(zone.isin(ENT),L20/(280*step),L20/45)
day=cc[(cc.y==1)&(cc.sun_elev>0)].case_id
d=cc[cc.case_id.isin(day)].copy()
d['yr']=d.groupby('case_id').t.transform(lambda s: s[d.loc[s.index,'y']==1].min()).dt.year
def fit(D,cols):
    D=D.dropna(subset=cols); D=D[D.groupby('case_id').y.transform('sum')==1]; D=D[D.groupby('case_id').y.transform('size')>1]
    return ConditionalLogit(D.y,D[cols],groups=D.case_id).fit(disp=0),D
def grade(R,c1,c2): return np.select([R<c1,R<c2],[0,1],2)
def ev(D,c1,c2):
    D=D.copy(); gr=grade(D.R,c1,c2); D['중']=(gr==1).astype(int); D['상']=(gr==2).astype(int)
    m,u=fit(D,['중','상','rain']); return m,u
for zones,lab in [(ENT+EXT,'입구+출구'),(ENT,'입구만'),(EXT,'출구만')]:
  for src,col in [('ASOS','icsr'),('천리안','gk2a_icsr')]:
    dd=d[d.tunnel_zone.isin(zones)].copy(); dd['R']=ratio(dd[col],dd.tunnel_zone)
    dd=dd[(dd[col]>0)]
    dd=dd[dd.groupby('case_id').y.transform('sum')==1]
    n=dd[dd.y==1].case_id.nunique()
    print(f'\n===== {lab} · {src}: 사고 {n}건, 연도별', dd[dd.y==1].yr.value_counts().sort_index().to_dict())
    print('휘도비 분위:', dd.R.quantile([.1,.25,.5,.75,.9]).round(1).to_dict())
    try:
        m,u=ev(dd,10,15); ct=pd.crosstab(grade(u.R,10,15),u.y)
        print('①10/15 칸별(평소,사고):',{k:tuple(v) for k,v in ct.iterrows()}, '배수',np.exp(m.params).round(2).to_dict(),'p',m.pvalues.round(3).to_dict())
    except Exception as e: print('① 실패',e)
    tr=dd[dd.yr<=2024]; te=dd[dd.yr==2025]
    cands=[5,6,8,10,12,15,20,30,50]; best=None
    for i,c1 in enumerate(cands):
        for c2 in cands[i+1:]:
            try: m,u=ev(tr,c1,c2)
            except Exception: continue
            g=grade(u.R,c1,c2)
            if min((g==k).sum() for k in (0,1,2))<20 or not np.isfinite(m.llf): continue
            if best is None or m.llf>best[0]: best=(m.llf,c1,c2)
    if best is None: print('② 후보 없음'); continue
    print('②23~24 최적:',best[1:])
    for name,(c1,c2) in [('①',(10,15)),('②',best[1:])]:
        try:
            m,u=ev(te,c1,c2); print(f'  2025 {name} {c1}/{c2}: llf {m.llf:.2f}, 배수',np.exp(m.params).round(2).to_dict(),'p',m.pvalues.round(3).to_dict(),'사고',u[u.y==1].case_id.nunique())
        except Exception as e: print('  2025',name,'실패',e)
    try: m0,_=fit(te,['rain']); print('  2025 기준선(비만) llf',round(m0.llf,2))
    except Exception as e: print(e)
